In [ ]:
# Load data, extract exchange rates, and normalize them
import pandas as pd
import numpy as np

# Load the CSV file into a pandas DataFrame
file_path = 'priceIndex_and_exchangeRate_series_2025_SP.csv'  # Replace with your CSV file path
data = pd.read_csv(file_path, index_col=0)

african_countries = [
    'DZA', 'AGO', 'BEN', 'BWA', 'BFA', 'BDI', 'CMR', 'CPV', 'CAF', 'TCD', 'COM', 'COG', 'CIV', 
    'COD', 'DJI', 'EGY', 'GNQ', 'ERI', 'SWZ', 'ETH', 'GAB', 'GMB', 'GHA', 'GIN', 'GNB', 'KEN', 
    'LBY', 'LSO', 'LBR', 'MDG', 'MWI', 'MLI', 'MRT', 'MUS', 'MAR', 'MOZ', 'NAM', 'NER', 'NGA', 
    'RWA', 'STP', 'SEN', 'SYC', 'SLE', 'SOM', 'ZAF', 'SSD', 'SDN', 'TZA', 'TGO', 'TUN', 'UGA', 
    'ZMB', 'ZWE'
]

def build_hybrid_index_df(
    start="1990-01-01",
    end="2023-01-01",
    countries=african_countries,
    df=data
):
    """
    Create tidy dataframe with columns:
      country | year | hybrid_index
    where hybrid_index = (NER / local_price_index), normalized to 100 at first valid obs.
    """

    start_date = pd.to_datetime(start)
    df.index = pd.to_datetime(df.index)
    df = df[(df.index >= start_date) & (df.index <= end)]
    
    records = []

    for country in countries:
        panusatls_col = f"{country}PANUSATLS"    # nominal exchange rate (LCU per USD)
        price_level_col = f"{country}NECONPRVTXN"  # local CPI or GDP deflator

        if panusatls_col not in df.columns or price_level_col not in df.columns:
            print(f"Missing columns for {country}")
            continue

        series =  df[panusatls_col] / df[price_level_col] 
        first_valid = series.first_valid_index()
        if first_valid is None:
            continue

        # normalize to 100 at first valid point
        series_norm = series / series.loc[first_valid] * 100

        for date, value in series_norm.items():
            if pd.isna(value):
                continue
            records.append({
                "country": country,
                "year": date.year,
                "hybrid_index": float(value)
            })

    tidy_df = pd.DataFrame(records).sort_values(["country", "year"]).reset_index(drop=True)
    return tidy_df

# Example usage:
# hybrid_df = build_hybrid_index_df(start="1990-01-01", end="2023-01-01")
# print(hybrid_df.head())

def contiguous_runs(years):
    """Return list of slices [start_idx, end_idx] for contiguous sequences (diff==1)."""
    years = np.asarray(years)
    if len(years) == 0:
        return []
    cuts = np.where(np.diff(years) != 1)[0] + 1
    starts = np.r_[0, cuts]
    ends   = np.r_[cuts, len(years)]
    return list(zip(starts, ends))  # half-open intervals

def compute_terminal_residuals(
    df,
    years=15,
    discount_rate=0.05,
    window_mode="two_non_overlapping"  # or "rolling"
):
    """
    Compute terminal residuals (PV of net normalized deviations) for fixed-horizon windows.
    
    df must have: 'country', 'year', 'hybrid_index'

    Returns DataFrame with:
      country, window_start, window_end, baseline_value, net_pv, loss
    """
    out = []
    for country, g in df.groupby("country"):
        g = g.sort_values("year").reset_index(drop=True)

        # contiguous runs of years
        runs = contiguous_runs(g["year"].values)
        for s, e in runs:
            run = g.iloc[s:e].reset_index(drop=True)
            n = len(run)
            if n < years:
                continue

            # choose window starts
            if window_mode == "two_non_overlapping":
                starts = list(range(0, n - years + 1, years))[:2]  # up to 2 per run
            elif window_mode == "rolling":
                starts = list(range(0, n - years + 1))
            else:
                raise ValueError("window_mode must be 'two_non_overlapping' or 'rolling'")

            for i0 in starts:
                win = run.iloc[i0:i0+years].copy()

                # skip windows with missing values in hybrid_index
                if win["hybrid_index"].isna().any():
                    continue

                base = float(win["hybrid_index"].iloc[0])

                # signed normalized deviation
                dev = (win["hybrid_index"].to_numpy() / base) - 1.0

                # discount factors to start of window (year-end cashflows)
                t = np.arange(1, years + 1, dtype=float)
                dfactors = (1.0 + discount_rate) ** (-t)

                net_pv = float(np.sum(dev * dfactors))
                loss   = max(net_pv, 0.0)

                out.append({
                    "country": country,
                    "window_start": int(win["year"].iloc[0]),
                    "window_end": int(win["year"].iloc[-1]),
                    "baseline_value": base,
                    "net_pv": net_pv,
                    "loss": loss
                })

    return pd.DataFrame(out)

# Example usage:
# residuals = compute_terminal_residuals(df, years=15, discount_rate=0.05,
#                                        window_mode="two_non_overlapping")
# expected_loss = residuals["loss"].mean()
# prob_positive = (residuals["loss"] > 0).mean()

def annuity_factor(n_years=15, r=0.05):
    return (1 - (1 + r) ** -n_years) / r


In [ ]:
hybrid_df = build_hybrid_index_df(start="1990-01-01", end="2023-01-01")
residuals = compute_terminal_residuals(hybrid_df, years=15, discount_rate=0,
                                       window_mode="rolling")
prob_positive = (residuals["loss"] > 0).mean()
expected_loss = residuals["loss"].mean()
af = annuity_factor(n_years=15, r=0.1)
annual_premium = expected_loss / af
annual_premium, prob_positive


In [ ]:

residuals[residuals.loss <10].loss.mean()/af
# residuals.loss.mean() / af
residuals[residuals.loss >= 10]
af, annual_premium, expected_loss, prob_positive

In [ ]:
print(residuals['net_pv'].describe())
residuals['net_pv'].hist(bins=50)

In [ ]:
import matplotlib.pyplot as plt

# Create a pivot table with years as index and countries as columns
pivot_df = hybrid_df.pivot(index='year', columns='country', values='hybrid_index')

# Plot all countries as separate lines
fig, ax = plt.subplots(figsize=(14, 8))
pivot_df.plot(ax=ax, alpha=0.7, linewidth=1)

# Customize the plot
ax.set_title("Hybrid Exchange Rate Indices by Country (1990-2023)", fontsize=14)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Hybrid Index (Normalized to 100)", fontsize=12)
ax.grid(True, alpha=0.3)

# Move legend outside the plot area
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', ncol=2, fontsize=8)

# Adjust layout to accommodate legend
plt.tight_layout()
plt.show()

In [ ]:
# Create a new dataframe with a row for each country in african_countries
country_years = pd.DataFrame({'country': african_countries})

# Initialize first_year and last_year columns with NaN
country_years['first_year'] = np.nan
country_years['last_year'] = np.nan

# For each country in african_countries, find the first and last year in hybrid_df
for country in african_countries:
    country_data = hybrid_df[hybrid_df['country'] == country]
    if not country_data.empty:
        country_years.loc[country_years['country'] == country, 'first_year'] = country_data['year'].min()
        country_years.loc[country_years['country'] == country, 'last_year'] = country_data['year'].max()

country_years
country_years['first_year'].hist(bins=range(1990, 2025))

In [ ]:
dza = residuals[residuals.country == 'DZA'].sort_values(['window_start'])
dza
hybrid_df.to_csv('hybrid_index.csv', index=False)


In [ ]:
import matplotlib.pyplot as plt
r_values = np.arange(0.0, 0.5, 0.01)
af_values = [annuity_factor(n_years=20, r=r) for r in r_values]

plt.plot(r_values, af_values, marker='o')
plt.xlabel('Discount Rate (r)')
plt.ylabel('Annuity Factor (15 years)')
plt.title('Annuity Factor vs Discount Rate')
plt.grid(True)
plt.show()

In [ ]:
annuity_factor(n_years=15, r=0.00001)